# Model Evaluation and Deployment

This notebook evaluates the final model against a benchmark heuristic and deploys the model to SageMaker.

In [12]:
import pandas as pd
from sklearn.metrics import classification_report
from sagemaker import get_execution_role
from sagemaker.xgboost.model import XGBoostModel
import sagemaker

# Load test data from S3
test_df = pd.read_csv("s3://sagemaker-us-east-1-520193275417/final_project/feature_engineer/Xy_test.csv")

# Adjust this column name as needed
label_column = "label"  # Replace with your actual label column name

# Split into features and labels
X_test = test_df.drop(columns=[label_column, "record_id"])
y_test = test_df[label_column]

# Set up SageMaker session
sagemaker_session = sagemaker.Session()
role = get_execution_role()

# Use your trained XGBoost model artifact in S3
model_artifact = "s3://sagemaker-us-east-1-520193275417/final_project/output/xgboost-tuning-1750543902/output/model.tar.gz"

# Create SageMaker XGBoost model
xgb_model = XGBoostModel(
    model_data=model_artifact,
    role=role,
    framework_version="1.3-1",
    sagemaker_session=sagemaker_session
)

from sagemaker.predictor import Predictor

# Reuse the existing endpoint (don't deploy again)
predictor = Predictor(endpoint_name="xgb-model-endpoint", sagemaker_session=sagemaker_session)

# Predict using the deployed model (requires LibSVM input format)
from sklearn.datasets import dump_svmlight_file
import tempfile

# Convert X_test to libsvm format in memory
with tempfile.NamedTemporaryFile(mode="wb+", delete=False) as f:
    dump_svmlight_file(X_test, y_test, f)
    f.seek(0)
    libsvm_payload = f.read()

# Send to endpoint
# Set content type explicitly for SageMaker XGBoost endpoint
predictor.content_type = "text/libsvm"

# Send to endpoint
y_pred_raw = predictor.predict(libsvm_payload)

# Decode and convert predictions
if isinstance(y_pred_raw, bytes):
    y_pred_raw = y_pred_raw.decode("utf-8").strip().split("\n")

y_pred = [int(float(p)) for p in y_pred_raw]

# Print evaluation report
print("Final Model Performance:\n")
print(classification_report(y_test, y_pred))

Final Model Performance:

              precision    recall  f1-score   support

           0       0.42      1.00      0.59      9232
           1       0.00      0.00      0.00     12674

    accuracy                           0.42     21906
   macro avg       0.21      0.50      0.30     21906
weighted avg       0.18      0.42      0.25     21906



/opt/conda/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/conda/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/conda/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [15]:
# Load heuristic predictions from S3 (ensure this CSV aligns with y_test order)
heuristic_preds_df = pd.read_csv("s3://sagemaker-us-east-1-520193275417/final_project/feature_engineer/heuristic_predictions.csv")

# Extract the predicted labels column
heuristic_preds = heuristic_preds_df["heuristic_pred"]

print("Heuristic Model Performance:\n")
print(classification_report(y_test, heuristic_preds))

Heuristic Model Performance:

              precision    recall  f1-score   support

           0       1.00      1.00      1.00      9232
           1       1.00      1.00      1.00     12674

    accuracy                           1.00     21906
   macro avg       1.00      1.00      1.00     21906
weighted avg       1.00      1.00      1.00     21906



## Comparison Summary

Use the above reports to compare accuracy, precision, recall, and F1-score between your trained model and the heuristic baseline.

In [17]:
import sagemaker
from sagemaker.xgboost.model import XGBoostModel
from sagemaker.model_monitor import DataCaptureConfig
from sagemaker import get_execution_role
import sys
import time

sys.path.append('../config')
import config

# SageMaker setup
sagemaker_session = sagemaker.Session()
role = get_execution_role()
bucket = config.S3_BUCKET

# Update with your actual model S3 path and entry point script
model_artifact = f's3://{bucket}/final_project/output/xgboost-tuning-1750543902/output/model.tar.gz'
s3_data_capture_path = f's3://{bucket}/final_project/monitoring/data-capture'

# Create and deploy model
xgb_model = XGBoostModel(
    model_data=model_artifact,
    role=role,
    framework_version="1.5-1",
    sagemaker_session=sagemaker_session
)

endpoint_name = f"final-model-endpoint-{int(time.time())}"
predictor = xgb_model.deploy(
    initial_instance_count=1,
    instance_type='ml.m5.large',
    endpoint_name=endpoint_name,
    data_capture_config=DataCaptureConfig(
        enable_capture=True,
        destination_s3_uri=s3_data_capture_path,
        csv_content_types=['text/csv']
    )
)

print(f"Model deployed. Endpoint name: {endpoint_name}")

------!Model deployed. Endpoint name: final-model-endpoint-1750546481


In [18]:
from sagemaker.serializers import CSVSerializer
import numpy as np

predictor.serializer = CSVSerializer()
sample_input = X_test.iloc[:5]
results_raw = predictor.predict(sample_input.values)
predictions_prob = [float(result[0]) for result in results_raw]
predictions_binary = (np.array(predictions_prob) > 0.5).astype(int)

print("Raw probabilities from endpoint:", predictions_prob)
print("Binary predictions (threshold > 0.5):", predictions_binary.tolist())
print("Actual labels for this sample:", y_test.iloc[:5].values.tolist())

Raw probabilities from endpoint: [5.019500349590089e-06, 0.9999912977218628, 0.000156313632032834, 0.9999396800994873, 5.019500349590089e-06]
Binary predictions (threshold > 0.5): [0, 1, 0, 1, 0]
Actual labels for this sample: [0, 1, 0, 1, 0]
